In [9]:
import os
import http.client
import getpass

conn = http.client.HTTPSConnection("api.evolus.ai")

api_key = os.environ.get("EVOLUS_API_KEY") or getpass.getpass("Evolus API key: ")
if not api_key:
    raise RuntimeError("An Evolus API key is required to call the agent API.")

headers = {
  "X-ApiKey": api_key,
  "X-Application-Id": "50d9b290-0675-45ec-a49c-885d89e597ec"
}

conn.request(
    "GET",
    "/api/v1/agents/775d884d-11fc-4937-bdab-8d49024e3050",
    headers=headers,
)

response = conn.getresponse()
print(response.read().decode())

conn.close()

{"agentId":"775d884d-11fc-4937-bdab-8d49024e3050","applicationId":"50d9b290-0675-45ec-a49c-885d89e597ec","name":"Emergency Room Digital Twin","description":"Simulates and optimizes emergency department operations using real-time data and predictive insights.","tagId":"775d884d","prompt":"You support emergency department leaders by maintaining a virtual representation of emergency room operations, capacity, patient flow, staffing, resources, and potential bottlenecks. You integrate and interpret approved real-time and historical data, clearly distinguishing observed facts, forecasts, assumptions, and simulated scenarios. You help users model changes such as staffing levels, bed availability, triage policies, ambulance arrivals, treatment times, and surges, then explain likely operational impacts, risks, and trade-offs. You provide concise dashboards, alerts, forecasts, scenario comparisons, and actionable recommendations prioritized by urgency and expected effect. You never replace clin

In [10]:
# import subprocess
# import time
# import webbrowser
# from pathlib import Path
# from urllib.error import HTTPError, URLError
# from urllib.request import urlopen

# _current = Path.cwd().resolve()
# _candidates = [_current / "digital-twin"]
# if _current.name.lower() == "digital-twin":
#     _candidates.insert(0, _current)
# _candidates.extend(parent / "digital-twin" for parent in _current.parents)
# project_root = next((path for path in _candidates if path.is_dir()), None)
# if project_root is None:
#     raise FileNotFoundError(f"Could not find a digital-twin folder from {_current}")

# ports = (4173, 5173)
# app_url = None
# for port in ports:
#     try:
#         with urlopen(f"http://127.0.0.1:{port}/api/simulation-control", timeout=2):
#             app_url = f"http://127.0.0.1:{port}"
#             break
#     except HTTPError as error:
#         if error.code != 404:
#             raise
#     except URLError:
#         continue

# if app_url is None:
#     print(f"Starting the app from {project_root}...")
#     server = subprocess.Popen("npm run dev", cwd=str(project_root), shell=True)
#     deadline = time.monotonic() + 60
#     while time.monotonic() < deadline:
#         if server.poll() is not None:
#             raise RuntimeError("The app server exited before becoming ready. Check the server output; run 'npm run install:ci' first if dependencies are not installed.")
#         for port in ports:
#             try:
#                 with urlopen(f"http://127.0.0.1:{port}/api/simulation-control", timeout=2):
#                     app_url = f"http://127.0.0.1:{port}"
#                     break
#             except HTTPError as error:
#                 if error.code != 404:
#                     raise
#             except URLError:
#                 continue
#         if app_url:
#             break
#         time.sleep(1)
#     if app_url is None:
#         raise TimeoutError("The app did not become ready within 60 seconds. Check the server output.")

# print(f"App is ready at {app_url}")
# webbrowser.open(app_url)
# print("The dashboard is opening. Once it loads, run the next cell to start the simulation.")


In [11]:

import subprocess
import time
import webbrowser
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

_current = Path.cwd().resolve()
_candidates = [_current / "digital-twin"]

if _current.name.lower() == "digital-twin":
    _candidates.insert(0, _current)

_candidates.extend(parent / "digital-twin" for parent in _current.parents)

project_root = next(
    (path for path in _candidates if path.is_dir()), None
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not find a digital-twin folder from {_current}"
    )

ports = (4173, 5173)


def find_app():
    for port in ports:
        try:
            with urlopen(
                f"http://127.0.0.1:{port}/",
                timeout=2
            ) as response:
                if response.status == 200:
                    return f"http://127.0.0.1:{port}"

        except (HTTPError, URLError, TimeoutError, OSError):
            continue

    return None


app_url = find_app()

if app_url is None:
    print(f"Starting the app from {project_root}...", flush=True)

    server = subprocess.Popen(
        ["cmd.exe", "/c", "npm.cmd run dev"],
        cwd=str(project_root)
    )

    deadline = time.monotonic() + 60

    while time.monotonic() < deadline:
        if server.poll() is not None:
            raise RuntimeError(
                "The app server exited. Check the npm output."
            )

        app_url = find_app()

        if app_url:
            break

        time.sleep(1)

    if app_url is None:
        raise TimeoutError(
            "The dashboard did not respond within 60 seconds."
        )

print(f"App is ready at {app_url}", flush=True)
webbrowser.open(app_url)
print(
    "The dashboard is opening. "
    "Once it loads, run the next cell to start the simulation."
)


App is ready at http://127.0.0.1:4173
The dashboard is opening. Once it loads, run the next cell to start the simulation.


In [12]:
import json
import time
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

TWIN_APIS = [
    "http://127.0.0.1:4173/api/simulation-state",
    "http://127.0.0.1:5173/api/simulation-state",
]
live = None
active_api = None
for twin_api in TWIN_APIS:
    try:
        with urlopen(twin_api, timeout=5) as response:
            live = json.load(response)
        active_api = twin_api
        break
    except HTTPError as error:
        if error.code not in {404, 503}:
            raise
    except URLError:
        continue

if live is None or active_api is None:
    raise ConnectionError("Could not read a live simulation. Start the app, open its dashboard, and try again.")

snapshot = live.get("snapshot", {})
run_id = snapshot.get("runId")
if not run_id:
    raise ValueError("The live snapshot does not contain a simulation run ID.")

control_api = active_api.rsplit("/", 1)[0] + "/simulation-control"
command_request = Request(
    control_api,
    data=json.dumps({"action": "start", "runId": run_id}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urlopen(command_request, timeout=10) as response:
    command = json.load(response)

deadline = time.monotonic() + 30
while time.monotonic() < deadline:
    status_url = f"{control_api}?id={command['id']}"
    with urlopen(status_url, timeout=10) as response:
        result = json.load(response)
    if result["status"] != "pending":
        break
    time.sleep(0.5)
else:
    raise TimeoutError("The start request is still queued. Keep the dashboard open and check its state before retrying.")

if result["status"] != "applied":
    raise RuntimeError(f"Simulation start was rejected: {result.get('message', 'No reason provided')}")
print(result.get("message", "Simulation started."))


Simulation is running.


In [13]:
import json
import re
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def read_live_digital_twin(candidates=None):
    candidates = candidates or [
        "http://127.0.0.1:4173",
        "http://127.0.0.1:5173",
        "http://localhost:4173",
        "http://localhost:5173",
    ]
    for candidate in candidates:
        try:
            req = Request(candidate, headers={"User-Agent": "Notebook-Live-Reader"})
            with urlopen(req, timeout=5) as response:
                html = response.read().decode("utf-8", errors="replace")
            if "Emergency" in html or "Ops copilot" in html or "Uniq" in html:
                return html
        except (URLError, HTTPError, TimeoutError, ValueError):
            continue
    return None

live_html = read_live_digital_twin()
if live_html:
    print("Live digital-twin page detected.")
    cleaned = re.sub(r"<script.*?</script>", " ", live_html, flags=re.S | re.I)
    cleaned = re.sub(r"<style.*?</style>", " ", cleaned, flags=re.S | re.I)
    cleaned = re.sub(r"<[^>]+>", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    patient_names = sorted(set(re.findall(r"\b[A-Z][a-z]+(?: [A-Z][a-z]+)+\b", cleaned)))
    room_markers = sorted(set(re.findall(r"\b(?:Resus|Exam|Acute|Flex|CT|Ventilator|Isolation|Blood)\b[^<]*", cleaned, flags=re.I)))
    print("Detected patient names:", patient_names[:25])
    print("Detected room/resource markers:", room_markers[:25])
    print("HTML preview:", cleaned[:1000])
else:
    print("No live digital-twin app responded at http://127.0.0.1:4173 or http://127.0.0.1:5173. Falling back to project inventory.")
    _current = Path.cwd().resolve()
    _candidates = [_current / "digital-twin"]
    if _current.name.lower() == "digital-twin":
        _candidates.insert(0, _current)
    _candidates.extend(parent / "digital-twin" for parent in _current.parents)
    project_root = next((path for path in _candidates if path.is_dir()), None)
    if project_root is None:
        raise FileNotFoundError(
            f"Could not find a digital-twin folder from the current directory: {_current}"
        )
    _ignored_dirs = {
        ".git", "node_modules", ".next", ".wrangler", ".vinext",
        ".sites-runtime", "coverage", "dist", "__pycache__",
    }
    _text_extensions = {
        ".ts", ".tsx", ".js", ".jsx", ".mjs", ".cjs", ".json", ".md",
        ".css", ".html", ".py", ".yml", ".yaml", ".toml", ".sh", ".sql",
        ".txt", ".xml", ".svg", ".mts", ".cts",
    }
    _max_text_bytes = 1_000_000
    _secret_patterns = [
        re.compile(r"(?i)(api[_-]?key|access[_-]?token|client[_-]?secret|password)(\s*[=:]\s*)([\"']?)[^\s\"',;]+"),
        re.compile(r"(?i)(authorization\s*:\s*)(bearer\s+)[A-Za-z0-9._~+/=-]+"),
    ]
    project_files = []
    for folder, dirnames, filenames in __import__("os").walk(project_root):
        dirnames[:] = sorted(name for name in dirnames if name not in _ignored_dirs)
        for filename in sorted(filenames):
            path = Path(folder) / filename
            if filename.startswith(".") and filename.lower() not in {".gitignore", ".eslintrc"}:
                continue
            try:
                stat = path.stat()
                content = None
                if path.suffix.lower() in _text_extensions and stat.st_size <= _max_text_bytes:
                    content = path.read_text(encoding="utf-8", errors="replace")
                    for pattern in _secret_patterns:
                        content = pattern.sub(r"\1\2[REDACTED]", content)
                project_files.append({
                    "path": path.relative_to(project_root).as_posix(),
                    "extension": path.suffix.lower() or "[no extension]",
                    "size_bytes": stat.st_size,
                    "content": content,
                })
            except OSError:
                continue
    print(f"Project folder: {project_root}")
    print(f"Files inventoried: {len(project_files)}")
    print(f"Text contents collected: {sum(record['content'] is not None for record in project_files)}")
    print("Inventory preview:")
    for record in project_files[:20]:
        print(f"{record['path']} ({record['size_bytes']:,} bytes)")


Live digital-twin page detected.
Detected patient names: ['Amina Patel', 'Avery Emergency', 'Deterioration Deterministic', 'Emergency Operations Twin', 'Jonas Meyer', 'Kim Exam', 'Leila Hassan', 'Lind Trauma', 'Location Waiting Resources', 'Maya Rossi Complex', 'Noah Williams Severe', 'Nurse Anders Trauma', 'Nurse Kim Critical', 'Nurse Silva Emergency', 'Nurse Vega General Advisory', 'Okafor General', 'Owen Brooks', 'Ready Occupied Turnover Reserved Resus', 'Resources Clinical', 'Riley Chen Status', 'Seed Waiting', 'Silva Acute', 'Source Ambulance', 'Specialist Trauma Synthetic', 'State Version']
Detected room/resource markers: ['Resus 1 resus occupied 1 Noah Williams Severe respiratory failure 16 m remaining Avery · Kim Exam 1 exam occupied 4 Maya Rossi Complex ankle fracture 8 m remaining Okafor · Silva Acute 1 acute ready Ready for assignment Resus 2 resus cleaning · 3m Reserved for P-106 Exam 2 exam ready Ready for assignment Exam 3 exam ready Ready for assignment Flex bay acute re

In [14]:
import json
import time
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

# Edit these rows using IDs from the live choices printed below.
PAIRINGS = [
    {"patient_id": "P-101", "doctor_id": "D-03", "room_id": "R-03"},
]

TWIN_APIS = [
    "http://127.0.0.1:4173/api/simulation-state",
    "http://127.0.0.1:5173/api/simulation-state",
]
live = None
active_api = None
for twin_api in TWIN_APIS:
    try:
        with urlopen(twin_api, timeout=5) as response:
            live = json.load(response)
        active_api = twin_api
        break
    except HTTPError as error:
        if error.code not in {404, 503}:
            raise
    except URLError:
        continue

if live is None or active_api is None:
    raise ConnectionError("Could not reach a running digital-twin app with a published simulation. Start the app and keep the dashboard open.")

snapshot = live.get("snapshot", {})
run_id = snapshot.get("runId")
if not run_id:
    raise ValueError("The live snapshot does not contain a simulation run ID.")

print("Current choices:")
print(json.dumps({
    "waiting_patients": snapshot.get("waiting", []),
    "available_doctors": [member for member in snapshot.get("staff", []) if member["role"] == "doctor" and member["status"] == "available"],
    "ready_rooms": [room for room in snapshot.get("rooms", []) if room["status"] == "ready"],
}, indent=2))
print("Requested pairings:", json.dumps(PAIRINGS))

control_api = active_api.rsplit("/", 1)[0] + "/simulation-control"
command_request = Request(
    control_api,
    data=json.dumps({"action": "assign_selected", "runId": run_id, "pairings": PAIRINGS}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urlopen(command_request, timeout=10) as response:
    command = json.load(response)

deadline = time.monotonic() + 30
while time.monotonic() < deadline:
    status_url = f"{control_api}?id={command['id']}"
    with urlopen(status_url, timeout=10) as response:
        result = json.load(response)
    if result["status"] != "pending":
        break
    time.sleep(0.5)
else:
    raise TimeoutError("The assignment request is still queued. Check the dashboard before submitting another request.")

if result["status"] != "applied":
    raise RuntimeError(f"Assignment request was rejected: {result.get('message', 'No reason provided')}")
print("Assignment results:")
print(result.get("message", "Assignment request applied."))
print("An available nurse is selected automatically. Check the live snapshot to verify the assignments.")


Current choices:
{
  "waiting_patients": [
    {
      "id": "P-102",
      "name": "Sofia Karim",
      "severity": 2,
      "condition": "worsening",
      "waitMinutes": 12,
      "room": "resus",
      "resources": [
        "blood"
      ],
      "specialty": "General",
      "blockedReason": "no resus room; no available doctor",
      "deteriorations": 0
    },
    {
      "id": "P-105",
      "name": "Elias Berg",
      "severity": 3,
      "condition": "stable",
      "waitMinutes": 24,
      "room": "exam",
      "resources": [],
      "specialty": "General",
      "blockedReason": "no available doctor",
      "deteriorations": 0
    },
    {
      "id": "P-110",
      "name": "Ingrid Dahl",
      "severity": 3,
      "condition": "worsening",
      "waitMinutes": 20,
      "room": "exam",
      "resources": [
        "isolation"
      ],
      "specialty": "General",
      "blockedReason": "no available doctor",
      "deteriorations": 0
    }
  ],
  "available_doctors": [],


In [15]:
import json
import time
from urllib.error import HTTPError, URLError
from urllib.request import urlopen
from IPython.display import clear_output

TWIN_APIS = [
    "http://127.0.0.1:4173/api/simulation-state",
    "http://127.0.0.1:5173/api/simulation-state",
]
POLL_SECONDS = 1

print("Reading live simulation snapshots. Start the digital-twin app and press Run twin; interrupt this cell to stop.")
while True:
    live = None
    active_api = None
    for twin_api in TWIN_APIS:
        try:
            with urlopen(twin_api, timeout=5) as response:
                live = json.load(response)
            active_api = twin_api
            break
        except HTTPError as error:
            if error.code == 503:
                active_api = twin_api
                break
            if error.code != 404:
                raise
        except URLError:
            continue

    if live is None:
        clear_output(wait=True)
        if active_api:
            print(f"Waiting for the dashboard to publish its first simulation snapshot at {active_api}...")
        else:
            print(f"No digital-twin API found at {TWIN_APIS}; retrying...")
        time.sleep(POLL_SECONDS)
        continue

    snapshot = live["snapshot"]
    clear_output(wait=True)
    print(f"Published: {live['publishedAt']} | Simulation: {snapshot['simTime']} | Run: {snapshot['runId']} | State v{snapshot['stateVersion']}")
    print(json.dumps({
        "metrics": snapshot["metrics"],
        "waiting": snapshot["waiting"],
        "treating": snapshot["treating"],
        "inbound": snapshot["inbound"],
        "rooms": snapshot["rooms"],
        "staff": snapshot["staff"],
        "resources": snapshot["resources"],
    }, indent=2))
    time.sleep(POLL_SECONDS)


Published: 2026-10-16T17:40:36.381Z | Simulation: 09:15 | Run: RUN-42-MCI | State v107
{
  "metrics": {
    "waiting": 1,
    "treating": 2,
    "discharged": 7,
    "critical": 3,
    "occupiedRooms": 2,
    "roomUtilization": 29,
    "meanWait": 36.7,
    "peakQueue": 6,
    "deteriorationEvents": 39,
    "inbound": 0,
    "activeConflicts": 1,
    "offlineResources": 0,
    "resourcePressure": 71
  },
  "waiting": [
    {
      "id": "P-109",
      "name": "Owen Brooks",
      "severity": 1,
      "condition": "critical",
      "waitMinutes": 96,
      "room": "resus",
      "resources": [
        "ventilator"
      ],
      "specialty": "General",
      "blockedReason": "no resus room; Ventilators at capacity",
      "deteriorations": 15
    }
  ],
  "treating": [
    {
      "id": "P-107",
      "name": "Jonas Meyer",
      "severity": 1,
      "roomId": "R-04",
      "remainingMinutes": 1
    },
    {
      "id": "P-108",
      "name": "Leila Hassan",
      "severity": 1,
      "

KeyboardInterrupt: 